In [1]:
import numpy as np
import tensorflow as tf
import unicodedata
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model

# Normalize text
def normalize(text):
    return unicodedata.normalize('NFC', text)

# Dataset
data = [
    ("hello", "नमस्ते"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("thank you", "धन्यवाद"),
    ("good night", "शुभ रात्रि")
]

# Prepare data
inp_texts, tgt_texts = [], []
inp_chars, tgt_chars = set(), set()

for i, t in data:
    i = normalize(i)
    t = normalize(t)
    t = '\t' + t + '\n'   # start + end tokens
    
    inp_texts.append(i)
    tgt_texts.append(t)
    
    inp_chars.update(i)
    tgt_chars.update(t)

inp_chars = sorted(inp_chars)
tgt_chars = sorted(tgt_chars)

# Token dictionaries
in_tok = {c: i for i, c in enumerate(inp_chars)}
tg_tok = {c: i for i, c in enumerate(tgt_chars)}
rev_tg = {i: c for c, i in tg_tok.items()}

max_in = max(len(x) for x in inp_texts)
max_tg = max(len(x) for x in tgt_texts)

# One-hot encoding
enc_in = np.zeros((len(inp_texts), max_in, len(inp_chars)))
dec_in = np.zeros((len(inp_texts), max_tg, len(tgt_chars)))
dec_tar = np.zeros((len(inp_texts), max_tg, len(tgt_chars)))

for i, (inp, tgt) in enumerate(zip(inp_texts, tgt_texts)):
    for t, c in enumerate(inp):
        enc_in[i, t, in_tok[c]] = 1
    
    for t, c in enumerate(tgt):
        dec_in[i, t, tg_tok[c]] = 1
        if t > 0:
            dec_tar[i, t - 1, tg_tok[c]] = 1

# Model
latent = 256

# Encoder
enc_inputs = Input(shape=(None, len(inp_chars)))
_, state_h, state_c = LSTM(latent, return_state=True)(enc_inputs)
enc_states = [state_h, state_c]

# Decoder
dec_inputs = Input(shape=(None, len(tgt_chars)))
dec_lstm = LSTM(latent, return_sequences=True, return_state=True)
dec_outputs, _, _ = dec_lstm(dec_inputs, initial_state=enc_states)

dense = Dense(len(tgt_chars), activation='softmax')
dec_outputs = dense(dec_outputs)

# Training model
model = Model([enc_inputs, dec_inputs], dec_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
model.fit([enc_in, dec_in], dec_tar, epochs=300, verbose=0)

print("\nModel Summary:")
model.summary()



# Encoder model
enc_model = Model(enc_inputs, enc_states)

# Decoder model
h_in = Input(shape=(latent,))
c_in = Input(shape=(latent,))
dec_inputs_single = Input(shape=(None, len(tgt_chars)))

dec_outputs2, state_h2, state_c2 = dec_lstm(
    dec_inputs_single,
    initial_state=[h_in, c_in]
)

dec_outputs2 = dense(dec_outputs2)

dec_model = Model(
    [dec_inputs_single, h_in, c_in],
    [dec_outputs2, state_h2, state_c2]
)


def decode_sequence(input_seq):
    states = enc_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1, len(tgt_chars)))
    target_seq[0, 0, tg_tok['\t']] = 1

    decoded = ""

    while True:
        output_tokens, h, c = dec_model.predict(
            [target_seq] + states, verbose=0
        )

        idx = np.argmax(output_tokens[0, -1, :])
        char = rev_tg[idx]

        if char == '\n' or len(decoded) > max_tg:
            break

        decoded += char

        target_seq = np.zeros((1, 1, len(tgt_chars)))
        target_seq[0, 0, idx] = 1

        states = [h, c]

    return decoded


print("\nTranslations:")
for i in range(len(inp_texts)):
    print(inp_texts[i], "->", decode_sequence(enc_in[i:i+1]))


Model Summary:
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, None, 18)]   0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, None, 29)]   0           []                               
                                                                                                  
 lstm (LSTM)                    [(None, 256),        281600      ['input_1[0][0]']                
                                 (None, 256),                                                     
                                 (None, 256)]                                                     
                                                                              